# __PROJECT_NAME__

Notebook variant of the FLOX research scaffold. Same SMA(10/30)
crossover as `main.py`, split into cells so you can iterate on
indicators and parameters interactively.

In [ ]:
import os
import flox_py as flox

## Data

The bundled `data/btcusdt_sample.csv` ships 500 BTC/USDT 1m bars.
Set `__PROJECT_ENV__` to swap in your own CSV with columns
`timestamp,open,high,low,close,volume`.

In [ ]:
HERE = os.getcwd()
DEFAULT_CSV = os.path.join(HERE, "data", "btcusdt_sample.csv")
DATA_CSV = os.environ.get("__PROJECT_ENV__", DEFAULT_CSV)
DATA_CSV

## Symbols and strategy

Edit the strategy class below to plug in your own indicators and
signal logic. Anything that satisfies the `flox.Strategy` interface
works.

In [ ]:
registry = flox.SymbolRegistry()
btc = registry.add_symbol("exchange", "BTCUSDT", tick_size=0.01)
btc

In [ ]:
class __PROJECT_SLUG___strategy(flox.Strategy):
    """SMA(10/30) crossover."""

    def __init__(self, symbols):
        super().__init__(symbols)
        self.fast = flox.SMA(10)
        self.slow = flox.SMA(30)

    def on_trade(self, ctx, trade):
        fv = self.fast.update(trade.price)
        sv = self.slow.update(trade.price)
        if fv is None or sv is None or not self.slow.ready:
            return
        if fv > sv and ctx.is_flat():
            self.market_buy(0.01)
        elif fv < sv and ctx.is_flat():
            self.market_sell(0.01)

## Run a backtest

In [ ]:
bt = flox.BacktestRunner(registry, fee_rate=0.0004,
                         initial_capital=10_000)
bt.set_strategy(__PROJECT_SLUG___strategy([btc]))
stats = bt.run_csv(DATA_CSV, symbol="BTCUSDT")
stats

## Inspect results

In [ ]:
print(f"return : {stats['return_pct']:+.4f}%")
print(f"trades : {stats['total_trades']}  win={stats['win_rate']*100:.1f}%")
print(f"sharpe : {stats['sharpe']:.4f}")
print(f"max DD : {stats['max_drawdown_pct']:.4f}%")
print(f"net PnL: {stats['net_pnl']:.4f}")

## Next steps

- Replace the SMA crossover with your own indicators (`flox.EMA`,
  `flox.RSI`, `flox.ADX`, ...).
- Run a parameter sweep with `flox.Optimizer`.
- Move the same strategy class to live data via
  `flox_py.ccxt.CcxtBroker`.